# Homogeneous periodic normal modes — SciPy

This notebook is a stripped-down test case extracted from a larger normal-mode-analysis framework developed for coronal-loop simulations.

Here we consider a **static, homogeneous, adiabatic plasma** in one spatial dimension with **periodic boundary conditions**. Density and temperature perturbations are stored at cell centres and velocity perturbations on a staggered grid.

For perturbations proportional to \(e^{nt}\), the linearised equations are

\[
n\rho_1 = -\rho_0\frac{\partial v_1}{\partial s},
\]

\[
nv_1 = -\frac{1}{\rho_0}\left[
\frac{p_0}{\rho_0}\frac{\partial \rho_1}{\partial s}
+\frac{p_0}{T_0}\frac{\partial T_1}{\partial s}
\right],
\]

\[
nT_1 = -(\gamma-1)T_0\frac{\partial v_1}{\partial s}.
\]

For this homogeneous adiabatic problem the acoustic modes satisfy

\[
\omega_m = k_m c_s,\qquad
k_m=\frac{2\pi m}{L},\qquad
c_s=\sqrt{\frac{\gamma p_0}{\rho_0}}.
\]

The purpose of the notebook is numerical validation: the finite-difference eigenfrequencies are compared directly with this analytical result.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.sparse import diags, block_array, identity, csr_matrix
from scipy.sparse.linalg import eigs

gamma = 5/3
m_p = 1.6726217770e-24       # g
k_B = 1.3806488e-16          # erg K^-1

# Homogeneous equilibrium
n_e0 = 1e9                   # cm^-3
rho0 = m_p * n_e0            # g cm^-3
p0 = 0.165677856             # dyn cm^-2
Rtilde = k_B / m_p
T0 = p0 / (2 * rho0 * Rtilde)
c_s = np.sqrt(gamma * p0 / rho0)

# Periodic domain
L = 1e10                     # cm
N = 256
ds = L / N
s = np.arange(N) * ds

print(f"T0 = {T0:.3e} K")
print(f"c_s = {c_s/1e5:.2f} km/s")
print(f"L   = {L/1e8:.1f} Mm")

## Periodic staggered operators

The periodicity is built directly into the corner entries of the finite-difference matrices. `D_c2f` differentiates a cell-centred quantity onto the face grid, while `D_f2c` maps a face-centred derivative back to cell centres.

Because the domain is periodic, both grids contain \(N\) degrees of freedom.

In [ ]:
# centre -> face
D_c2f = diags(
    [-np.ones(N), np.ones(N)],
    [0, 1], shape=(N, N)
).tolil()
D_c2f[-1, 0] = 1.0
D_c2f = D_c2f.tocsr() / ds

# face -> centre
D_f2c = diags(
    [-np.ones(N), np.ones(N)],
    [-1, 0], shape=(N, N)
).tolil()
D_f2c[0, -1] = -1.0
D_f2c = D_f2c.tocsr() / ds

# centre -> face interpolation
I_c2f = diags(
    [0.5*np.ones(N), 0.5*np.ones(N)],
    [0, 1], shape=(N, N)
).tolil()
I_c2f[-1, 0] = 0.5
I_c2f = I_c2f.tocsr()

## Assemble the eigenvalue problem

The state vector is

\[
\mathbf{x}=(\rho_1,v_1,T_1)^T,
\]

and the discretised system is written as

\[
A\mathbf{x}=nB\mathbf{x}.
\]

For this test \(B\) is simply the identity.

In [ ]:
Z = csr_matrix((N, N))

A_rho_v = -rho0 * D_f2c

A_v_rho = -(p0 / rho0**2) * D_c2f
A_v_T   = -(p0 / (rho0*T0)) * D_c2f

A_T_v = -(gamma - 1) * T0 * D_f2c

A = block_array([
    [Z,         A_rho_v, Z],
    [A_v_rho,  Z,       A_v_T],
    [Z,         A_T_v,   Z]
], format="csr")

B = identity(3*N, format="csr")

print("Matrix size:", A.shape)

## Compare numerical and analytical acoustic frequencies

In [ ]:
nmodes = 5
nev = 6
rows = []
saved_modes = {}

for m in range(1, nmodes + 1):
    k = 2*np.pi*m/L
    omega_exact = k*c_s
    target = 1j*omega_exact

    eigvals, eigvecs = eigs(A, M=B, k=nev, sigma=target)
    j = np.argmin(np.abs(eigvals - target))

    eigval = eigvals[j]
    vec = eigvecs[:, j]
    omega_num = eigval.imag
    rel_err = abs(abs(omega_num) - omega_exact) / omega_exact

    rows.append((m, omega_num, omega_exact, rel_err))
    saved_modes[m] = (eigval, vec)

print(" m      omega_num [s^-1]      omega_exact [s^-1]    relative error")
for m, on, oe, err in rows:
    print(f"{m:2d}   {on: .8e}       {oe: .8e}        {err:.2e}")

## Representative eigenfunction

The global complex phase of an eigenvector is arbitrary. We rotate the eigenvector so that the largest-amplitude density point is real, and then normalise all perturbations by the maximum density amplitude.

In [ ]:
m = 1
eigval, vec = saved_modes[m]

rho1 = vec[:N].copy()
v1   = vec[N:2*N].copy()
T1   = vec[2*N:].copy()

j0 = np.argmax(np.abs(rho1))
phase = np.angle(rho1[j0])
rho1 *= np.exp(-1j*phase)
v1   *= np.exp(-1j*phase)
T1   *= np.exp(-1j*phase)

norm = np.max(np.abs(rho1))
rho1 /= norm
v1   /= norm
T1   /= norm

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(s/1e8, rho1.real, label=r"$\mathrm{Re}(\rho_1)$")
ax.plot(s/1e8, rho1.imag, "--", label=r"$\mathrm{Im}(\rho_1)$")
ax.set_xlabel("s [Mm]")
ax.set_ylabel("normalised amplitude")
ax.set_title(f"Density eigenfunction, m={m}")
ax.legend()
plt.show()

## Notes

This public example intentionally contains only the homogeneous periodic validation problem. The full research framework additionally treats spatially structured and dynamically evolving coronal-loop backgrounds and is not included here.